# DocMind AI - Intelligent Search & LOO Token Attribution Demo
This notebook demonstrates synonym query expansions, hybrid searches, and Dynamic Text Classification with explanations.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import json
from src.config import load_config
from src.search.hybrid_search import HybridSearchEngine
from src.models.classifier import ZeroShotClassifier
from src.utils.explainability import TextFeatureAttribution

## 1. Initialize Dataset & Search Engine

In [ ]:
corpus_path = '../data/sample/documents.json'
with open(corpus_path, 'r') as f:
    corpus = json.load(f)

config = load_config('../configs/config.yaml')
engine = HybridSearchEngine(
    corpus=corpus,
    encoder_model_name=config.encoder_model_name,
    cross_encoder_model_name=config.cross_encoder_model_name,
    hybrid_alpha=config.hybrid_alpha,
    query_expansion=config.query_expansion
)
print(f'Engine initialized with {len(corpus)} documents.')

## 2. Execute Hybrid Search & Reranking

In [ ]:
query = 'vision models and CNN segmentations'
results = engine.search(query, top_k=2)

for r in results:
    doc = r['document']
    print(f"Title: {doc['title']} (Category: {doc['category']})")
    print(f"Content: {doc['content']}")
    print(f"Hybrid Score: {r['hybrid_score']:.3f} | Rerank Score: {r.get('rerank_score', 0.0):.3f}")
    print('-' * 50)

## 3. Dynamic Classification & Word Attribution

In [ ]:
classifier = ZeroShotClassifier()
# Force fallback mock classification for immediate execution without downloading model weights
text = 'The Segment Anything Model is great for computer vision MRI scans.'
labels = ['Computer Vision', 'Generative AI', 'Time Series']

explainer = TextFeatureAttribution(classify_fn=classifier._mock_classify)
attributions = explainer.explain(text, target_label='Computer Vision', candidate_labels=labels)

print('Word Attribution Scores for Computer Vision Category:')
for word, score in attributions:
    print(f'Word: {word:<15} | Score: {score:+.3f}')